# DCC 고등부 예선 통합 답안

이 노트북은 Mission 1부터 Mission 3까지의 데이터 분석, 문제 정의, 모델 학습, 평가 및 오차 분석을 재현할 수 있도록 구성하였다. 실제 학습 전에는 경로와 실행 환경을 확인하고, 각 셀을 위에서 아래로 순서대로 실행한다.

## 실행 환경 설정

In [ ]:
# Colab에서 필요한 경우 한 번만 실행
# !pip -q install timm imagehash

from pathlib import Path
from collections import Counter, defaultdict
import json, math, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 로컬 경로. Colab에서는 전처리 데이터 경로로 수정한다.
DATA_ROOT = Path('/Users/yaeryung/Documents/water_level/291.수위 측정 데이터/01-1.정식개방데이터/Training')
IMAGE_DIR = DATA_ROOT / '01.원천데이터' / 'images'
LABEL_DIR = DATA_ROOT / '02.라벨링데이터' / 'labels'
OUTPUT_DIR = Path('dcc_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

assert IMAGE_DIR.exists(), IMAGE_DIR
assert LABEL_DIR.exists(), LABEL_DIR
print('IMAGE_DIR:', IMAGE_DIR)
print('LABEL_DIR:', LABEL_DIR)

# Mission 1 데이터 다운로드 및 분석

## Mission 1-1 파일 수와 대응 관계

Training 원천데이터와 라벨링데이터의 압축을 해제한 뒤 파일 수를 계산하고, 확장자를 제외한 파일명을 기준으로 이미지와 JSON의 일대일 대응 여부를 확인한다.

In [ ]:
image_paths = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
label_paths = sorted(LABEL_DIR.glob('*.json'))
image_by_stem = {p.stem: p for p in image_paths}
label_by_stem = {p.stem: p for p in label_paths}

print(f'원천 이미지: {len(image_paths):,}')
print(f'JSON 라벨: {len(label_paths):,}')
print(f'라벨 없는 이미지: {len(image_by_stem.keys() - label_by_stem.keys()):,}')
print(f'이미지 없는 라벨: {len(label_by_stem.keys() - image_by_stem.keys()):,}')

### Mission 1-1 결과

- 원천 이미지 308,085장
- JSON 라벨 308,085개
- 이미지 또는 라벨이 누락된 사례 0건

모든 이미지와 JSON이 파일명 기준으로 일대일 대응하였다.

## Mission 1-2 JSON 분석

각 JSON에서 실제 수위, 목자판 종류, 수위 구간, 방해물, 목자판 및 수면 annotation을 읽는다. 방해물은 한 이미지에 여러 항목이 기록될 수 있으므로 다중 집계한다.

In [ ]:
records, bad_json = [], []
for path in label_paths:
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        info = data.get('image', {})
        category = (data.get('categories') or [{}])[0]
        obstruction = info.get('obstruction') or []
        if isinstance(obstruction, str):
            obstruction = [obstruction]
        records.append({
            'stem': path.stem,
            'image_path': str(image_by_stem.get(path.stem, '')),
            'json_path': str(path),
            'mokja_type': category.get('mokjaType', path.stem[0]),
            'water_range': category.get('clslevel'),
            'waterlevel_m': float(info['waterlevel']),
            'climate': info.get('climate'),
            'device_type': info.get('deviceType'),
            'obstruction': obstruction,
        })
    except Exception as exc:
        bad_json.append((str(path), str(exc)))

metadata_df = pd.DataFrame(records)
print(f'정상 JSON: {len(metadata_df):,}, 오류 JSON: {len(bad_json):,}')
metadata_df.head()

In [ ]:
mokja_order = ['A', 'B', 'C', 'E', 'F', 'G', 'H', 'S']
mokja_counts = metadata_df['mokja_type'].value_counts().reindex(mokja_order, fill_value=0)
display(mokja_counts.rename('수량').to_frame())

obstruction_counter = Counter(
    value.strip() for values in metadata_df['obstruction'] for value in values
    if isinstance(value, str) and value.strip()
)
obstruction_counts = pd.Series(obstruction_counter).sort_values(ascending=False)
display(obstruction_counts.rename('수량').to_frame())

blank_obstruction = sum(
    1 for values in metadata_df['obstruction'] for value in values
    if isinstance(value, str) and not value.strip()
)
print('공백 방해물 값:', blank_obstruction)

### Mission 1-2 결과

목자판별 수량은 A 105,220개, B 37,266개, C 91,941개, E 23,506개, F 8,787개, G 10,977개, H 10,311개, S 20,077개였다. A와 C의 비중이 높고 F, G, H의 비중이 낮아 종류별 불균형이 존재하였다. 방해물 값이 공백인 사례 10건은 `해당없음`과 합치지 않고 별도 품질 문제로 기록하였다.

## Mission 1-3 목자판 바운딩박스 시각화

JSON의 `mokja` annotation에 저장된 상대 좌표를 영상 픽셀 좌표로 변환한다. 목자판 8종에서 각각 3장을 추출해 총 24장을 표시한다.

In [ ]:
from matplotlib.patches import Rectangle

def get_box(json_path, annotation_name='mokja'):
    data = json.loads(Path(json_path).read_text(encoding='utf-8'))
    width, height = data['image']['width'], data['image']['height']
    ann = next(a for a in data['annotations'] if a.get('name') == annotation_name)
    rel = ann['relative_coordinates']
    if isinstance(rel, str):
        rel = json.loads(rel)
    w, h = rel['width'] * width, rel['height'] * height
    x = rel['center_x'] * width - w / 2
    y = rel['center_y'] * height - h / 2
    return x, y, w, h

fig, axes = plt.subplots(8, 3, figsize=(15, 30))
for row, gauge_type in enumerate(mokja_order):
    samples = metadata_df[metadata_df.mokja_type == gauge_type].sample(3, random_state=SEED)
    for col, (_, sample) in enumerate(samples.iterrows()):
        ax = axes[row, col]
        img = Image.open(sample.image_path)
        x, y, w, h = get_box(sample.json_path)
        ax.imshow(img)
        ax.add_patch(Rectangle((x, y), w, h, fill=False, color='red', linewidth=3))
        ax.set_title(f'{gauge_type}  {sample.waterlevel_m:.2f} m')
        ax.axis('off')
plt.tight_layout()
plt.show()

## Mission 1-4 유사 이미지 그룹 기반 8 대 1 대 1 분할

단순 무작위 분할은 같은 장소나 연속 촬영 이미지가 Train과 Test에 동시에 포함되는 데이터 누수를 일으킬 수 있다. JSON과 이미지 EXIF에는 촬영 장소 및 세션 ID가 없으므로 perceptual hash를 이용해 시각적으로 가까운 이미지를 그룹화한다. 동일 그룹은 하나의 subset에만 배정한다.

이 방법은 실제 장소 ID를 완전히 복원하는 방법이 아니다. 따라서 그룹별 표본 이미지를 직접 확인하고, 해시 임계값을 바꾸었을 때 분할이 지나치게 변하지 않는지 함께 점검한다.

In [ ]:
# 전체 실행에는 시간이 걸리므로 결과를 CSV로 저장하고 재사용한다.
HASH_CACHE = OUTPUT_DIR / 'perceptual_hashes.csv'

def dhash64(path):
    with Image.open(path) as img:
        gray = img.convert('L').resize((9, 8), Image.Resampling.LANCZOS)
        arr = np.asarray(gray, dtype=np.int16)
    bits = arr[:, 1:] > arr[:, :-1]
    value = 0
    for bit in bits.ravel():
        value = (value << 1) | int(bit)
    return value

if HASH_CACHE.exists():
    hash_df = pd.read_csv(HASH_CACHE, dtype={'hash_hex': str})
else:
    rows = []
    for idx, row in metadata_df.iterrows():
        value = dhash64(row.image_path)
        rows.append({'stem': row.stem, 'hash_hex': f'{value:016x}'})
        if (idx + 1) % 10000 == 0:
            print(f'{idx + 1:,}/{len(metadata_df):,}')
    hash_df = pd.DataFrame(rows)
    hash_df.to_csv(HASH_CACHE, index=False)

hash_df.head()

In [ ]:
# Hamming 거리 기반 BK-tree와 Union-Find로 근접 이미지 그룹 생성
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b:
            return
        if self.size[a] < self.size[b]:
            a, b = b, a
        self.parent[b] = a
        self.size[a] += self.size[b]

def hamming(a, b):
    return (a ^ b).bit_count()

class BKTree:
    def __init__(self):
        self.root = None
    def add(self, value, index):
        node = [value, index, {}]
        if self.root is None:
            self.root = node
            return
        cur = self.root
        while True:
            d = hamming(value, cur[0])
            if d not in cur[2]:
                cur[2][d] = node
                return
            cur = cur[2][d]
    def query(self, value, radius):
        if self.root is None:
            return []
        result, stack = [], [self.root]
        while stack:
            cur = stack.pop()
            d = hamming(value, cur[0])
            if d <= radius:
                result.append(cur[1])
            lo, hi = d - radius, d + radius
            stack.extend(child for edge, child in cur[2].items() if lo <= edge <= hi)
        return result

merged = metadata_df.merge(hash_df, on='stem', how='left').reset_index(drop=True)
uf = UnionFind(len(merged))
HASH_RADIUS = 6

# 목자판 종류와 수위 구간이 같은 범위 안에서 유사 이미지를 검색한다.
for _, indices in merged.groupby(['mokja_type', 'water_range']).groups.items():
    tree = BKTree()
    for idx in indices:
        value = int(merged.at[idx, 'hash_hex'], 16)
        for neighbor in tree.query(value, HASH_RADIUS):
            uf.union(idx, neighbor)
        tree.add(value, idx)

merged['group_id'] = [uf.find(i) for i in range(len(merged))]
print('전체 이미지:', len(merged))
print('생성 그룹:', merged.group_id.nunique())
display(merged.groupby('group_id').size().describe())

In [ ]:
# 그룹이 겹치지 않으면서 목자판 종류와 수위 구간 비율을 유지하는 10개 fold 생성
from sklearn.model_selection import StratifiedGroupKFold

merged['stratum'] = merged.mokja_type.astype(str) + '_' + merged.water_range.astype(str)
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
merged['fold'] = -1

for fold, (_, val_idx) in enumerate(sgkf.split(merged, merged.stratum, groups=merged.group_id)):
    merged.loc[val_idx, 'fold'] = fold

merged['subset'] = np.where(merged.fold == 0, 'test', np.where(merged.fold == 1, 'validation', 'train'))
split_counts = merged.subset.value_counts().reindex(['train', 'validation', 'test'])
display(split_counts.rename('수량').to_frame())

# 누수 검사
group_subset_count = merged.groupby('group_id').subset.nunique()
assert group_subset_count.max() == 1, '동일 그룹이 여러 subset에 포함됨'
assert set(merged.stem) == set(metadata_df.stem)
print('그룹 누수 없음')

split_columns = ['stem', 'image_path', 'json_path', 'mokja_type', 'water_range',
                 'waterlevel_m', 'group_id', 'fold', 'subset']
merged[split_columns].to_csv(OUTPUT_DIR / 'group_split.csv', index=False)

### Mission 1-4 보고 항목

실행 후 Train, Validation, Test의 실제 수량과 목자판별 분포를 기록한다. 그룹 누수가 0건인지 확인하고, 각 그룹에서 대표 이미지를 추출해 같은 장소 또는 유사한 장면으로 묶였는지 육안 검증한다. 해시 임계값은 대표 그룹 검토 결과와 그룹 크기 분포를 근거로 확정한다.

In [ ]:
# 그룹 품질 육안 검증: 큰 그룹 12개에서 각 4장 표시
largest_groups = merged.groupby('group_id').size().sort_values(ascending=False).head(12).index
fig, axes = plt.subplots(12, 4, figsize=(14, 32))
for row, group_id in enumerate(largest_groups):
    samples = merged[merged.group_id == group_id].head(4)
    for col, ax in enumerate(axes[row]):
        if col < len(samples):
            sample = samples.iloc[col]
            ax.imshow(Image.open(sample.image_path))
            ax.set_title(f'group {group_id}')
        ax.axis('off')
plt.tight_layout()
plt.show()

## Mission 1-5 Colab 업로드 및 저장 구조

개별 파일 60만 개 이상을 Drive에서 직접 읽으면 입출력 병목과 누락 위험이 크다. Mac에서 이미지를 학습 입력 크기로 축소한 뒤 여러 ZIP 또는 shard로 묶어 업로드한다. Colab에서는 압축파일을 `/content/dataset`에 해제하고, 모델 가중치와 결과만 Drive에 저장한다.

권장 구조:

```text
MyDrive/DCC/water_level/
├── processed_shards
├── splits
├── checkpoints
└── results
```

원본 이미지를 직접 사용하는 로컬 분석과 Colab용 경량 학습 데이터 생성 절차를 구분해 재현성을 유지한다.

# Mission 2 수위 추정 문제 정의

## Mission 2-1 문제 정의

수위 추정 문제를 **목자판 종류를 조건으로 사용하는 연속값 회귀 문제**로 정의한다. 입력은 수위 측정 이미지와 테스트 파일명 첫 글자에서 얻는 목자판 종류이고, 출력은 미터 단위 수위이다. 정답이 연속적인 숫자이며 평가 지표가 RMSE이므로 회귀가 직접적인 문제 정의이다.

## Mission 2-2 수위 추정 핵심 요소

- 목자판 전체 영역과 종류
- 목자판의 숫자와 눈금
- 수면선 위치
- 목자판과 수면이 만나는 지점
- 목자판별 측정 범위
- 촬영 배율과 기울기

최종 모델은 전체 영상에서 특징을 추출하고 목자판 종류 임베딩을 결합한다. 이 구조는 테스트 시 허용된 이미지와 파일명 첫 글자만 사용한다.

## Mission 2-3 난이도 요인

반사광, 달빛, 그림자, 야간, 안개, 탁류, 거센 물살과 와류는 수면선을 흐리게 만든다. 수풀, 사람, 부유물과 부착물은 목자판을 가린다. 표면 손상과 오염, 낮은 해상도, 작은 목자판, 촬영 각도 차이는 숫자와 눈금 인식을 어렵게 만든다. Mission 3-3에서는 이 조건별 RMSE와 고오차 사례를 함께 분석한다.

# Mission 3 모델 개발 및 성능 분석

## Mission 3-1 후보 모델 탐색과 선택

| 후보 | 장점 | 한계 | 역할 |
|---|---|---|---|
| ResNet50 | 구조가 단순하고 널리 검증됨 | 계산량 대비 효율이 낮을 수 있음 | 기준 모델 |
| EfficientNet B0 | 적은 파라미터로 정확도와 속도의 균형 | 작은 목자판 정보가 축소될 수 있음 | 최종 선택 |
| ConvNeXt Tiny | 강한 이미지 특징 추출 성능 | 메모리와 학습 시간이 증가 | 고성능 비교 후보 |

최종 모델로 EfficientNet B0를 선택한다. Colab Pro에서 반복 실험하기에 비교적 가볍고, 사전학습 가중치를 활용할 수 있으며, 목자판 종류 임베딩을 결합한 회귀 헤드를 추가하기 쉽기 때문이다. 공정한 비교를 위해 같은 분할, 입력 크기, epoch, optimizer와 평가 지표를 사용한다.

## 학습 데이터 전처리

Train에는 밝기·대비·약한 회전·흐림 등 수위 의미를 훼손하지 않는 증강만 적용한다. 상하 반전과 큰 회전은 수위 방향을 바꾸므로 사용하지 않는다. Validation과 Test에는 resize와 normalize만 적용한다. 테스트 JSON의 수위, 바운딩박스와 수면 위치는 모델 입력에 사용하지 않는다.

In [ ]:
# Colab 학습 코드
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
TYPE_TO_IDX = {name: idx for idx, name in enumerate(mokja_order)}

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomRotation(3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class WaterLevelDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row.image_path).convert('RGB')
        return {
            'image': self.transform(image),
            'type_idx': torch.tensor(TYPE_TO_IDX[row.mokja_type], dtype=torch.long),
            'target': torch.tensor(row.waterlevel_m, dtype=torch.float32),
            'stem': row.stem,
        }

class ConditionalRegressor(nn.Module):
    def __init__(self, backbone_name='efficientnet_b0'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feature_dim = self.backbone.num_features
        self.type_embedding = nn.Embedding(len(TYPE_TO_IDX), 8)
        self.head = nn.Sequential(
            nn.Linear(feature_dim + 8, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1)
        )
    def forward(self, image, type_idx):
        image_feature = self.backbone(image)
        type_feature = self.type_embedding(type_idx)
        return self.head(torch.cat([image_feature, type_feature], dim=1)).squeeze(1)

train_df = merged[merged.subset == 'train'].copy()
val_df = merged[merged.subset == 'validation'].copy()
test_df = merged[merged.subset == 'test'].copy()

train_loader = DataLoader(WaterLevelDataset(train_df, train_tf), batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(WaterLevelDataset(val_df, eval_tf), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(WaterLevelDataset(test_df, eval_tf), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

model = ConditionalRegressor('efficientnet_b0').to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.MSELoss()
print('device:', device)

In [ ]:
def evaluate(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            image = batch['image'].to(device)
            type_idx = batch['type_idx'].to(device)
            prediction = model(image, type_idx).cpu().numpy()
            target = batch['target'].numpy()
            rows.extend(zip(batch['stem'], target, prediction))
    result = pd.DataFrame(rows, columns=['stem', 'actual_m', 'predicted_m'])
    result['squared_error'] = (result.actual_m - result.predicted_m) ** 2
    return result, math.sqrt(result.squared_error.mean())

best_rmse = float('inf')
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        image = batch['image'].to(device)
        type_idx = batch['type_idx'].to(device)
        target = batch['target'].to(device)
        prediction = model(image, type_idx)
        loss = criterion(prediction, target)
        loss.backward()
        optimizer.step()

    val_result, val_rmse = evaluate(model, val_loader)
    print(f'epoch {epoch + 1:02d}  validation RMSE {val_rmse:.4f} m')
    if val_rmse < best_rmse:
        best_rmse = val_rmse
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_efficientnet_b0.pt')

## Mission 3-2 전체 및 목자판별 RMSE

Validation으로 선택한 최종 가중치를 한 번만 Test에 적용한다. 전체 RMSE와 A, B, C, E, F, G, H, S별 RMSE를 meter 단위로 계산한다.

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pt', map_location=device))
test_result, overall_rmse = evaluate(model, test_loader)
test_result = test_result.merge(metadata_df[['stem', 'mokja_type', 'obstruction']], on='stem', how='left')

type_rmse = test_result.groupby('mokja_type').squared_error.mean().pow(0.5).reindex(mokja_order)
print(f'전체 Test RMSE: {overall_rmse:.4f} m')
display(type_rmse.rename('RMSE_m').to_frame())

test_result.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

## Mission 3-3 오차 사례 분석

절대오차가 큰 표본을 확인하고 방해물, 목자판 종류, 수위 구간과의 관계를 분석한다. 모델이 목자판 대신 배경을 참고했는지, 수면선이 반사·가림·탁류로 불명확했는지, 특정 목자판 종류의 표본 부족이 영향을 주었는지 검토한다.

In [ ]:
test_result['absolute_error'] = (test_result.actual_m - test_result.predicted_m).abs()
worst = test_result.nlargest(24, 'absolute_error')
display(worst[['stem', 'mokja_type', 'actual_m', 'predicted_m', 'absolute_error', 'obstruction']])

fig, axes = plt.subplots(6, 4, figsize=(16, 22))
for ax, (_, row) in zip(axes.ravel(), worst.iterrows()):
    image_path = image_by_stem[row.stem]
    ax.imshow(Image.open(image_path))
    ax.set_title(f'{row.mokja_type}  true {row.actual_m:.2f} / pred {row.predicted_m:.2f}\nerror {row.absolute_error:.2f} m')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 방해물별 RMSE: 다중 라벨을 행 단위로 펼쳐 계산
obstruction_eval = test_result[['stem', 'squared_error', 'obstruction']].explode('obstruction')
obstruction_eval['obstruction'] = obstruction_eval.obstruction.fillna('정보없음').replace('', '공백')
obstruction_rmse = obstruction_eval.groupby('obstruction').squared_error.mean().pow(0.5).sort_values(ascending=False)
display(obstruction_rmse.rename('RMSE_m').to_frame())

# 최종 결론 작성 지침

모델 실행 후 다음 항목을 실제 결과로 채운다.

1. 그룹 분할의 Train, Validation, Test 수량과 그룹 누수 검사 결과
2. 비교한 모델별 Validation RMSE와 학습 시간
3. 최종 모델의 전체 Test RMSE
4. 목자판 8종별 RMSE
5. 방해물별 RMSE와 대표 실패 사례
6. 한계와 개선 방향

실행하지 않은 수치나 확인하지 않은 성능은 기재하지 않는다.